# 10장 실습 ③ — 시드만 바꾼다

**TensorFlow 판**

여기서 진짜 문제가 드러납니다. **같은 코드입니다. 시드만 다릅니다.**

## 10.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 10.1 실험대 — 표시된 자리를 기억하기

순차열은 대부분 잡음입니다. **한 자리**에만 신호가 있고, 두 번째 채널이
그 자리를 표시합니다. **그 신호의 부호**를 맞힙니다.

표시된 자리가 앞쪽이므로, 그 값을 **끝까지 여러 걸음 들고 가야** 합니다.

In [ ]:
# 표시된 자리의 값을 끝까지 들고 가야 풀리는 문제.
x, y = data.memory_task(4000, length=80, seed=42)
sl = data.split(x, y, val_ratio=0.15, test_ratio=0.15, seed=42)
print(sl.summary())

fig, ax = plt.subplots(figsize=(8.5, 2.6))
ax.plot(x[0, :, 0], lw=1.2, label="값")
ax.plot(x[0, :, 1], lw=1.2, ls="--", label="표시")
ax.set_xlabel("걸음"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title(f"표시된 자리의 부호를 맞혀야 합니다 (정답 {y[0]})")
plt.show()

## 10.2 학습 함수 — 여기만 판마다 다릅니다

**PyTorch 판에 `Recurrent` 래퍼가 하나 더 있는 것**에 주목하십시오.
PyTorch의 순환 층은 (출력 전체, 마지막 상태)를 돌려주므로,
Keras의 기본 동작(마지막 것만)과 맞추려면 감싸야 합니다.

In [ ]:
import tensorflow as tf

L_ = tf.keras.layers

def _recurrent(kind, units=32):
    return {"rnn": L_.SimpleRNN, "lstm": L_.LSTM, "gru": L_.GRU}[kind](units)

def train_seq(kind, sp, length, lr=0.001, seed=42, epochs=25):
    """분류: 표시된 자리의 부호 맞히기. (시험 정확도)

    이 함수 하나만 판마다 다르다. 아래의 모든 실험 셀은 세 판이 같다.
    """
    dlbook.set_seed(seed)
    ls = [L_.Input(shape=(length, 2))]
    if kind == "dnn":
        ls += [L_.Flatten(), L_.Dense(64, activation="relu")]
    else:
        ls += [_recurrent(kind)]
    ls += [L_.Dense(2, activation="softmax")]
    m = tf.keras.Sequential(ls)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss="sparse_categorical_crossentropy")
    m.fit(sp.x_train, sp.y_train, epochs=dlbook.smoke.epochs(epochs),
          batch_size=64, verbose=0)
    return metrics.accuracy(sp.y_test, m.predict(sp.x_test, verbose=0).argmax(1))

def train_forecast(kind, sp, lr=0.003, seed=42, epochs=30):
    """회귀: 다음 값 맞히기. (MAE, 파라미터 수)"""
    dlbook.set_seed(seed)
    ls = [L_.Input(shape=(sp.x_train.shape[1], 1))]
    if kind == "dnn":
        ls += [L_.Flatten(), L_.Dense(64, activation="relu")]
    else:
        ls += [_recurrent(kind)]
    ls += [L_.Dense(1)]
    m = tf.keras.Sequential(ls)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="mse")
    m.fit(sp.x_train, sp.y_train, validation_data=(sp.x_val, sp.y_val),
          epochs=dlbook.smoke.epochs(epochs), batch_size=64, verbose=0)
    return metrics.mae(sp.y_test, m.predict(sp.x_test, verbose=0).reshape(-1)), \
        m.count_params()

## 10.5 실험 ③ — 시드만 바꿔 봅니다

여기서 진짜 문제가 드러납니다.

In [ ]:
# 실험 ③ — 학습률까지 고정하고 **시드만** 바꾼다. 여기서 진짜 문제가 나온다.
L = 80
seeds = [0, 42] if dlbook.smoke.is_smoke() else [0, 1, 42, 7]

print(f"길이 {L}, 학습률 0.001, 25 epoch. 초기 가중치 시드만 바꿉니다.")
print(f"{'':<12}" + "".join(f"{'seed ' + str(sd):>10}" for sd in seeds)
      + f"{'평균':>9}{'표준편차':>10}")
for k in ("rnn", "lstm", "gru"):
    row = [train_seq(k, sl, L, lr=0.001, seed=sd) for sd in seeds]
    print(f"{k:<12}" + "".join(f"{v:>10.3f}" for v in row)
          + f"{np.mean(row):>9.3f}{np.std(row):>10.3f}")
    dlbook.record(f"ch10_seedstd_{k}", float(np.std(row)))

print()
print("→ **같은 코드입니다. 시드만 다릅니다.** 0.46에서 1.00까지 갑니다.")
print("→ 한 번 돌린 결과로는 아무것도 알 수 없습니다. (7장 §7.7)")
print("→ 순환 신경망을 비교할 때는 학습률을 훑고, 시드를 여러 개 돌리고,")
print("   **평균과 표준편차를 함께 보고**하십시오.")

## 정리

- **★ 같은 코드에서 시드만 바꿔도 0.46에서 1.00까지 갑니다.**
  RNN은 길이만큼 깊은 신경망이고, 같은 가중치가 길이만큼 곱해집니다.
  초기값이 경계의 어느 쪽에 떨어지느냐가 결과를 가릅니다.
- 그러니 **한 번 돌린 결과로는 아무것도 알 수 없습니다.**
  학습률을 훑고, 시드를 여러 개 돌리고, **평균과 표준편차를 보고**하십시오.

### 연습

1. LSTM과 SimpleRNN에 대해 **시드 10개**를 돌려 평균과 표준편차를 구하십시오.
   어느 쪽이 더 안정적입니까.
2. `clipnorm=1.0`(기울기 절단)을 켜면 표준편차가 줄어듭니까.
3. **[열린 문제]** 본문 §10.5에서 GRU가 길이 80에서 학습되지 않았습니다.
   학습률 5종 × 시드 4종 × epoch 3종을 시험했으나 0.48을 넘지 못했습니다.
   원인을 찾아 저장소 Issues에 남겨 주십시오.